<a href="https://colab.research.google.com/github/MDS2024K/SwAV-Network/blob/CNN_Baseline/Mod%C3%A8le_CNN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [ ]:
import torch, torchvision
from torch import nn
from torch import optim
from torchvision.transforms import ToTensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
num_batch = 64

## Getting the Data

In [ ]:
T = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor()])

train_data = torchvision.datasets.MNIST('mnist_data', train=True, download=True, transform=T)
val_data = torchvision.datasets.MNIST('mnist_data', train=False, download=True, transform=T)

train_dl = torch.utils.data.DataLoader(train_data, batch_size = num_batch)
val_dl = torch.utils.data.DataLoader(val_data, batch_size = num_batch)

## Create the model

In [ ]:
def create_leNet():
    model = nn.Sequential(
        nn.Conv2d(1, 6, 5, padding=2),
        nn.ReLU(),
        nn.AvgPool2d(2, stride=2),
        nn.Conv2d(6, 16, 5, padding=0),
        nn.ReLU(),
        nn.AvgPool2d(2, stride=2),
        nn.Flatten(),
        nn.Linear(400, 120),
        nn.ReLU(),
        nn.Linear(120, 84),
        nn.ReLU(),
        nn.Linear(84, 10)
    )
    return model

## Validation du modèle

In [ ]:

# la validation utilise l'ensemble de données de validation pour calculer
# le nombre d'images prédites correctement parmi le nombre total d'images.

def validate(model, data):
    total = 0
    correct = 0
    for i, (images, labels) in enumerate(data):
        images = images.cuda()
        x = model(images)
        value, pred = torch.max(x,1)
        pred = pred.data.cpu()
        total += x.size(0)
        correct += torch.sum(pred == labels)
    return correct*100./total

## Training Function

In [ ]:
import copy

def train(num_epoch=3, lr=1e-3, device="cpu"):
    accuracies = []
    cnn = create_leNet().to(device)
    cec = nn.CrossEntropyLoss()  # fonction loss
    optimizer = optim.Adam(cnn.parameters(), lr=lr) # Adam optimiseur
    max_accuracy = 0
    for epoch in range(num_epoch):
        for i, (images, labels) in enumerate(train_dl):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            pred = cnn(images)
            loss = cec(pred, labels)
            loss.backward()
            optimizer.step()
        accuracy = float(validate(cnn, val_dl))  # Calcul d'accuracy des données de validation après l'entraînement du modèle
        accuracies.append(accuracy)
        if accuracy > max_accuracy:
            best_model = copy.deepcopy(cnn)
            max_accuracy = accuracy
            print("Saving Best Model with Accuracy: ", accuracy)
        print('Epoch:', epoch+1, "Accuracy :", accuracy, '%')
    plt.plot(accuracies)  # Traçer le graphique des accuracies
    return best_model

## Disponibilité des GPU

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
    print("No Cuda Available")
device

In [ ]:
# Tester la fonction train avec 30 epoch et taux d'apprentissage lr (=0.001 par défaut)
num_epoch=30
lenet = train(num_epoch, device=device)